### RAG with Dockling & Langchain

In [1]:
!pip install -q qdrant-client sentence-transformers docling groq docling-hierarchical-pdf


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

from sentence_transformers import SentenceTransformer

C:\Users\Shraddha\PycharmProjects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SOURCE = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/AtliqAI_HR_Policies.pdf"

def load_and_chunk(source: str):
    """
    Load AND hierarchically chunk a document in a single step using
    LangChain's Docling integration.

    export_type=ExportType.DOC_CHUNKS tells DoclingLoader to run Docling's
    HierarchicalChunker under the hood and hand back one LangChain
    `Document` per chunk (instead of one Document for the whole file).
    Each Document's headings/provenance info lives in `metadata["dl_meta"]`.
    """
    loader = DoclingLoader(
        file_path=source,
        export_type=ExportType.DOC_CHUNKS,
    )
    return loader.load()

doc_chunks = load_and_chunk(SOURCE)
print(f"Total chunks: {len(doc_chunks)}")

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-08-24 18:01:22,451 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-24 18:01:22,461 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-24 18:01:22,473 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Shraddha\PycharmProjects\RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-24 18:01:22,473 [RapidOCR] main.py:50: Using C:\Users\Shraddha\PycharmProjects\RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-24 18:01:22,738 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-24 18:01:22,739 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-24 18:01:22,741 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users

Total chunks: 44


In [4]:
# Inspect a raw DocChunk
sample = doc_chunks[2]
print(f"content  : {sample.page_content[:200]}…")
print(f"headings : {sample.metadata.get('dl_meta', {}).get('headings')}")

content  : Probation Period
All new employees at AtliqAI are placed on a probation period of 6 months from the date of joining. During this period, either party may terminate the employment with a notice period …
headings : ['Probation Period']


In [5]:
from langchain_core.documents import Document
def convert_chunk(doc_chunk) -> Document:
    """
    Convert a Docling DocChunk into a plain dict.

    headings   → list preserved as-is
    content    → paragraph text
    chunk_text → breadcrumb + content  (what gets embedded)
    """
    headings   = doc_chunk.metadata.get('dl_meta', {}).get('headings') or []
    content    = doc_chunk.page_content.strip()
    breadcrumb = " > ".join(headings)
    chunk_text = f"{breadcrumb}\n\n{content}" if breadcrumb else content

    return Document(
        page_content=chunk_text,
        metadata={"headings": headings, "content": content},
    )

chunks = [convert_chunk(c) for c in doc_chunks]

In [6]:
for chunk in chunks[:3]:
    print("─" * 60)
    print(f"headings   : {chunk.metadata['headings']}")
    print(f"content    : {chunk.metadata['content'][:200]}…")

────────────────────────────────────────────────────────────
headings   : ['AtliqAI HR Policies']
content    : AtliqAI HR Policies
AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compe…
────────────────────────────────────────────────────────────
headings   : ['Offer and Joining Formalities']
content    : Offer and Joining Formalities
Upon acceptance of an offer letter, candidates must complete the joining formalities within the stipulated date mentioned in the offer. The HR team will share a prejoinin…
────────────────────────────────────────────────────────────
headings   : ['Probation Period']
content    : Probation Period
All new employees at AtliqAI are placed on a probation period of 6 months from the date of joining. During this period, either party may terminate the employment with a notice period …


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7098.83it/s]


In [8]:
!rmdir /s /q "\tmp\my_qdrant"

In [9]:
from langchain_qdrant import QdrantVectorStore
COLLECTION_NAME = "docs"

vectorstore = QdrantVectorStore.from_documents(
    chunks,
    embedding=embeddings,
    path="/tmp/my_qdrant",
    collection_name=COLLECTION_NAME,
    force_recreate=True,   # fresh collection each run, like recreate_collection() did
)
print(f"Indexed {len(chunks)} chunks into Qdrant collection '{COLLECTION_NAME}'.")


Indexed 44 chunks into Qdrant collection 'docs'.


In [10]:
info = vectorstore.client.get_collection(COLLECTION_NAME)
print(f"Points     : {info.points_count}")

Points     : 44


In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [12]:
results = retriever.invoke("What is the leave policy?")
for r in results:
    print(f"[{r.metadata['headings']}]")
    print(f"  {r.metadata['content'][:200]}…\n")


[['Leave Without Pay']]
  Leave Without Pay
Employees who exhaust all available leave balances may apply for leave without pay (LWP). LWP must be approved by the reporting manager and HR. More than 10 days of LWP in a financia…

[['Casual Leave']]
  Casual Leave
Every confirmed employee is entitled to 12 casual leaves per calendar year, credited at 1 leave per month. Casual leave can be availed for personal errands, minor illness, or unplanned ab…

[['AtliqAI HR Policies']]
  AtliqAI HR Policies
AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compe…

[['Sick Leave']]
  Sick Leave
Employees are entitled to 10 sick leaves per calendar year. Sick leave can be availed in case of illness, hospitalisation, or medical procedures. A medical certificate from a registered pra…

[['Earned Leave']]
  Earned Leave
Employees accrue earned leave at the rate of 1.25 days per m

In [13]:

from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """You are a helpful HR assistant.
Answer the user's question using ONLY the context provided below.
If the context does not contain enough information, say so — do not make things up.
Always cite the section name when referencing specific information."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])


In [14]:
def format_docs(docs: list[Document]) -> str:
    parts = []
    for i, doc in enumerate(docs, 1):
        parts.append(f"[Source {i}]\n{doc.metadata['content']}")
    return "\n\n---\n\n".join(parts)

In [15]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

GROQ_MODEL = "openai/gpt-oss-safeguard-20b"

llm = ChatGroq(model=GROQ_MODEL, temperature=0.2)   # Low = factual; High = creative

# LCEL chain: retriever -> format_docs -> prompt -> llm -> string output
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def rag(query: str):
    """
    End-to-end RAG pipeline, now expressed as a LangChain LCEL chain:
      1. Retrieve  — retriever fetches the top-k relevant chunks
      2. Format    — format_docs turns them into a context block
      3. Generate  — prompt + ChatGroq produce the final answer
    """
    docs = retriever.invoke(query)          # kept separately so we can also return sources
    answer = rag_chain.invoke(query)
    return answer, format_docs(docs)


In [18]:
answer, context = rag("How many casual leaves am I entitled to?")
print(answer)
print(f"{250*'='}")
print(f"\n\nSOURCES:\n {context}")

You are entitled to **12 casual leaves per calendar year** (credited at 1 leave per month). These leaves cannot be carried forward to the next year and will lapse on December 31st. (Source 1)


SOURCES:
 [Source 1]
Casual Leave
Every confirmed employee is entitled to 12 casual leaves per calendar year, credited at 1 leave per month. Casual leave can be availed for personal errands, minor illness, or unplanned absences. A maximum of 3 consecutive casual leaves can be taken at a time. Casual leaves cannot be carried forward to the next calendar year and lapse on December 31st.

---

[Source 2]
Sick Leave
Employees are entitled to 10 sick leaves per calendar year. Sick leave can be availed in case of illness, hospitalisation, or medical procedures. A medical certificate from a registered practitioner is mandatory for sick leave of more than 2 consecutive days. Unused sick leaves up to a maximum of 10 can be carried forward to the following year.

---

[Source 3]
Leave Without Pay
Employee